# Quiz 5: arrival comparison graph

Upload the two simulation downloads from WUI-Go and get the time versus cumulative arrivals graph.

1. In WUI-Go, open **Results**, select the run, choose **Download simulation outputs**. Do this for the baseline run and for your own case. Do not unzip or rename.
2. Run the cell below. In the upload box, select **both** ZIP files: the baseline first, then your own case.

The graph is saved as `arrival_comparison.png`. Open the Files panel on the left to download it for your PDF.


In [ ]:
# Run this cell. When the upload box appears, select BOTH ZIP files you downloaded
# from WUI-Go: first the baseline run, then your own case (hold Ctrl to pick two files).
# You do not need to rename or unzip them.

import io, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()

def read_final_agents(name, data):
    """Return final_agents.csv as a DataFrame from a WUI-Go zip or a csv upload."""
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            members = [m for m in z.namelist() if m.endswith("final_agents.csv")]
            if not members:
                raise ValueError(f"{name}: no final_agents.csv inside this zip")
            with z.open(members[0]) as f:
                return pd.read_csv(f)
    return pd.read_csv(io.BytesIO(data))

if len(uploaded) != 2:
    raise SystemExit(f"You uploaded {len(uploaded)} file(s). Run this cell again and select "
                     "exactly two files: the baseline ZIP first, then your own case ZIP.")

# Label by upload order; a file name containing 'baseline' or 'own' overrides the order.
names = list(uploaded.keys())
labels = ["Baseline", "Own case"]
for i, n in enumerate(names):
    low = n.lower()
    if "baseline" in low:
        labels[i] = "Baseline"
    elif "own" in low:
        labels[i] = "Own case"
if labels[0] == labels[1]:
    labels = ["Baseline", "Own case"]

runs = []
for label, name in zip(labels, names):
    df = read_final_agents(name, uploaded[name])
    if "agent_status" not in df.columns or "arrival_time" not in df.columns:
        raise ValueError(f"{name}: this is not the original WUI-Go output "
                         f"(columns found: {list(df.columns)[:6]} ...)")
    runs.append((label, name, df))

plt.figure(figsize=(6, 4))
for label, name, df in runs:
    arrived = df[df["agent_status"] == "arrive"]
    t = arrived["arrival_time"].sort_values()
    plt.step(t / 60, range(1, len(t) + 1), where="post", label=label)
    n_total, n_arr = len(df), len(arrived)
    last = t.max() / 60 if n_arr else float("nan")
    print(f"{label} ({name}): {n_arr} of {n_total} vehicles arrived; "
          f"{n_total - n_arr} did not; last arrival at {last:.1f} min")

plt.xlabel("Elapsed time (min)")
plt.ylabel("Cumulative vehicles arrived")
plt.xlim(left=0); plt.ylim(bottom=0)
plt.legend()
plt.tight_layout()
plt.savefig("arrival_comparison.png", dpi=200)
plt.show()
print("Saved arrival_comparison.png (left panel > Files) for your PDF.")
